In [1]:
import pandas as pd
import numpy as np
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

df = pd.read_csv("data/clean/ab_clean.csv")
base = df.loc[df["group"] == "control", "converted"].mean()
n_per_group = df["group"].value_counts().min()
users_per_day = len(df) / df["date"].nunique()
power_calc = NormalIndPower()

print(f"Baseline conversion: {base*100:.2f}%")
print(f"Users per group: {n_per_group:,}")
print(f"Users per day (both groups): {users_per_day:,.0f}")

Baseline conversion: 12.04%
Users per group: 145,274
Users per day (both groups): 12,634


In [2]:
h = power_calc.solve_power(nobs1=n_per_group, alpha=0.05, power=0.8, ratio=1)
mde_rate = np.sin(np.arcsin(np.sqrt(base)) + h / 2) ** 2
print(f"MDE: {(mde_rate - base)*100:.3f} pp  ({(mde_rate/base - 1)*100:.2f}% relative)")

MDE: 0.340 pp  (2.83% relative)


In [4]:
rows = []
for lift in [0.01, 0.02, 0.03, 0.05, 0.10]:
    es = proportion_effectsize(base * (1 + lift), base)
    power_now = power_calc.power(effect_size=abs(es), nobs1=n_per_group, alpha=0.05, ratio=1)
    n_needed = power_calc.solve_power(effect_size=abs(es), power=0.8, alpha=0.05, ratio=1)
    rows.append({
        "relative_lift": f"{lift*100:.0f}%",
        "new_rate_%": round(base * (1 + lift) * 100, 2),
        "power_of_this_test_%": round(power_now * 100, 1),
        "users_needed_per_group": int(np.ceil(n_needed)),
        "days_needed": round(2 * n_needed / users_per_day, 0)})
power_table = pd.DataFrame(rows)
import os
os.makedirs("outputs", exist_ok=True)
power_table.to_csv("outputs/power_table.csv", index=False)
power_table

,relative_lift,new_rate_%,power_of_this_test_%,users_needed_per_group,days_needed
0,1%,12.16,16.9,1151908,182.0
1,2%,12.28,51.0,289207,46.0
2,3%,12.40,84.4,129081,20.0
3,5%,12.64,99.9,46859,7.0
4,10%,13.24,100.0,11955,2.0
